# 📗 대화와 승인 대기를 저장하고 다시 이어갑니다

**대화를 이어가는 서비스 제작 상담 에이전트**를 만듭니다. 고객이 만들려는 서비스의 요구사항을 상담하고, 대화를 SQLite에 저장해 프로그램을 다시 시작해도 이어갑니다.

별도의 **안내문 승인 그래프**에서는 사람의 검토 대기를 저장하고, 재시작 후 승인·반려를 처리합니다.

<img src="images/03_persistence_overview.png" width="1050" style="background:white" alt="서비스 제작 상담 에이전트와 별도의 안내문 승인 그래프입니다. 각각 대화와 검토 대기를 SQLite에 저장하고 같은 DB 파일과 thread_id로 이어갑니다.">

## 이번 교안에서 배울 내용

1. 메모리 저장과 파일 저장을 구분하고 상담 그래프의 State·노드·엣지를 준비합니다.
2. 서로 다른 두 대화를 SQLite에 분리해 저장합니다.
3. 새 커널에서 대화를 먼저 조회하고 후속 질문을 보냅니다.
4. 고객 안내문의 승인 대기를 파일에 저장합니다.
5. 새 커널에서 승인과 반려를 재개합니다.

🖐️ 마지막 함께 따라하기에서는 날씨와 검색 Tool을 사용하는 여행 플래너의 계획·승인 대기를 저장하고 복원합니다.

**3절과 5절 시작에서 커널을 재시작만 합니다.** `Restart & Run All`을 누르지 말고 해당 절의 준비 셀부터 실행하세요. 위에서 아래로 전체 실행하는 것만으로는 프로그램 종료 후 복원을 확인한 것이 아닙니다.

학생용 결과는 `output/`, 정답 결과는 `output/answers/`에 따로 저장합니다.

환경은 앞 교안과 같습니다. 본문의 추가 패키지는 `langgraph-checkpoint-sqlite`이며, 처음 실행할 때 상담에 GPT를 3회 호출합니다. 정답의 선택 확인 셀은 별도 1회입니다. 함께 따라하기는 `requests·langchain`을 추가로 사용하며 날씨·검색·GPT 호출이 별도로 발생합니다.

## 1. 저장할 State와 상담 그래프를 준비합니다

**체크포인트**는 실행 단계별 State와 다음 작업을 담은 기록이고, **체크포인터**는 이를 저장하고 읽는 기능입니다. SqliteSaver로 파일에 저장하면 커널을 재시작해도 기록이 남습니다.

| 저장 방식 | 보관 위치 | 커널 재시작 뒤 |
|---|---|---|
| InMemorySaver | 체크포인터 객체의 메모리 | 사라짐 |
| 파일 경로를 쓰는 SqliteSaver | SQLite 파일 | 같은 파일에서 조회 가능 |

<img src="images/06_checkpoint_storage.png" width="1050" style="background:white" alt="State와 다음 작업 및 중단 정보를 파일에 보관하고 그래프 정의를 다시 준비해 복원합니다.">

DB에는 **State(메시지 포함)와 다음 작업·중단 정보**가 저장됩니다. 커널을 재시작하면 Python 변수와 객체는 사라지므로 **State 스키마·노드·엣지, 모델과 DB 연결은 다시 준비**합니다. [공식 체크포인터 문서](https://docs.langchain.com/oss/python/langgraph/checkpointers)

### 저장 파일과 모델을 준비합니다

`SqliteSaver.from_conn_string(파일경로)`는 DB 연결과 체크포인터를 준비합니다. 저장 테이블은 필요할 때 자동 생성되므로 `setup()`을 따로 호출하지 않습니다. `with`가 끝나면 연결이 닫힙니다. `:memory:`는 파일이 아니므로 재시작 후 복원할 수 없습니다.

In [ ]:
# 대화와 검토 대기를 보관할 SQLite 파일 위치 준비
from pathlib import Path
from langgraph.checkpoint.sqlite import SqliteSaver

material_dir = Path(".")
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
chat_db = output_dir / "lesson03_chat.sqlite"
approval_db = output_dir / "lesson03_approval.sqlite"
print("대화 저장 파일:", chat_db.resolve())
print("승인 저장 파일:", approval_db.resolve())

In [ ]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

# 앞 단원과 같은 환경 설정을 읽고 답변 모델을 준비
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)

### 1-1. State: 상담 근거와 대화를 담습니다

policy는 대화 시작 때 넣고 유지합니다. messages는 `add_messages`로 새 질문과 답변을 합칩니다. 딕셔너리의 **키는 필드 이름**, **값은 해당 필드에 저장할 데이터**입니다.

In [ ]:
# 대화마다 유지할 상담 근거와 메시지 누적 규칙 정의
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    policy: str  # 답변의 근거로 유지할 서비스 제작 상담 정책
    messages: Annotated[list[AnyMessage], add_messages]  # 같은 대화의 질문과 답변을 누적

In [ ]:
# 학습용 가상 제작사 정책
policy_text = """
[scope] 첫 상담에서는 서비스 목적, 이용 대상, 필수 기능을 확인합니다.
[estimate] 비용과 제작 일정은 요구사항 검토 후 안내합니다. 첫 상담에서 확정하지 않습니다.
[reservation] 예약 서비스는 공간 수, 운영 시간, 예약 단위, 취소 기준을 추가로 확인합니다.
[catalog] 상품 소개 서비스는 상품 수, 사진·설명 자료, 구매 연결 방식부터 확인합니다.
""".strip()

### 1-2. 노드 answer: 이전 대화를 읽고 답합니다

전체 그래프의 `START -> answer -> END`에서 답변 작업을 만듭니다. 노드는 State의 policy·messages를 읽고 새 AI 메시지만 반환합니다.

In [ ]:
# 정책과 이전 대화를 읽어 새 답변을 만드는 체인과 노드 정의
from IPython.display import Markdown, display
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "서비스 제작 의뢰 상담 담당자입니다. 대화의 마지막 질문에 한국어로 짧게 답하세요. "
     "이전 질문의 서비스 맥락을 유지하고, 지금 확인할 정보를 최대 3개로 정리하세요. "
     "제공 정책에 없는 비용이나 일정을 만들지 말고 해당 정책 ID를 [reservation]처럼 표시하세요. "
     "실제 접수나 계약이 완료되었다고 말하지 마세요.\n정책:\n{policy}"),
    MessagesPlaceholder("messages"),
])
answer_chain = answer_prompt | llm

def answer(state: ChatState):
    """저장된 정책과 대화를 읽고 새 답변 한 개를 반환합니다."""
    reply = answer_chain.invoke({"policy": state["policy"], "messages": state["messages"]})
    return {"messages": [reply]}

### 1-3. 엣지: 답변 작업을 연결합니다

여기서는 builder에 노드와 엣지를 등록합니다. 다음 절에서 DB 연결을 열고 `compile(checkpointer=...)`로 저장 기능을 연결합니다.

In [ ]:
from langgraph.graph import END, START, StateGraph

chat_builder = StateGraph(ChatState)
chat_builder.add_node("answer", answer)
# 노드 사이의 실행 흐름 연결
chat_builder.add_edge(START, "answer")
chat_builder.add_edge("answer", END)

## 2. 두 대화를 나누어 저장합니다

같은 DB에서도 **thread_id가 다르면 서로 다른 State**를 사용합니다. A는 예약 서비스, B는 상품 소개 서비스 상담입니다. 같은 사용자도 여러 thread를 가질 수 있습니다.

아래 코드에서 저장과 실행은 다음 순서로 연결됩니다.

1. `compile(checkpointer=checkpointer)`: 그래프에 SQLite 저장·조회 기능을 연결합니다.
2. `config`의 `thread_id`: 사용할 대화 기록을 지정합니다.
3. `invoke(입력, config)`: 해당 대화의 이전 State에 새 입력을 반영해 실행하고, 단계별 체크포인트를 저장합니다. 기록이 없으면 입력으로 시작합니다.

**입력 딕셔너리는 처리할 데이터이고, `config`는 실행 설정입니다.** `invoke()`만 `with`로 감싸도 체크포인터가 자동 연결되지는 않으므로, 아래처럼 컴파일·실행·조회를 같은 `with` 안에 둡니다.

셀을 다시 실행하면 저장된 대화는 조회만 합니다. 새 연습을 시작하려면 이 절과 3절의 thread_id를 함께 바꾸세요. 기존 DB를 지울 필요는 없습니다.

In [ ]:
# thread_id는 사용자 ID가 아니라 각각의 대화를 구분하는 ID
config_a = {"configurable": {"thread_id": "service-reservation-v1"}}
config_b = {"configurable": {"thread_id": "service-catalog-v1"}}

In [ ]:
from IPython.display import Image, display

# 입력 딕셔너리의 policy에는 상담 근거, messages에는 첫 질문을 전달
first_questions = [
    (config_a, "스터디룸 4개를 예약하는 모바일 웹을 만들고 싶어요. 무엇부터 정리할까요?"),
    (config_b, "수제 가방 20개를 소개하고 구매 링크로 연결하는 웹을 만들고 싶어요. 어떤 자료가 필요한가요?"),
]
with SqliteSaver.from_conn_string(str(chat_db)) as checkpointer:
    chat_graph = chat_builder.compile(checkpointer=checkpointer)
    display(Image(chat_graph.get_graph().draw_mermaid_png()))
    for config, question in first_questions:
        saved = chat_graph.get_state(config)
        if not saved.values:
            chat_graph.invoke({"policy": policy_text, "messages": [HumanMessage(content=question)]}, config)
        saved = chat_graph.get_state(config)
        print("대화 ID:", config["configurable"]["thread_id"])
        for message in saved.values["messages"]:
            print(message.type + ":")
            display(Markdown(message.text))
        print("다음 노드:", saved.next)

A와 B에 서로의 질문이 섞이지 않았는지 확인하세요. 완료한 실행의 `next`는 빈 튜플 `()`입니다.

이 예제에서는 **with 안에서 컴파일·실행·조회**합니다. with가 끝나면 DB 연결만 닫히고 저장 파일은 남습니다. 연결이 닫힌 그래프를 다시 호출하지 말고, 같은 파일에 재연결해 컴파일합니다.

## 3. 커널을 재시작하고 대화를 이어갑니다

**커널을 재시작만 한 뒤 아래 준비 셀부터 실행하세요.** 저장된 내용을 먼저 읽으므로 1·2절의 입력 셀을 다시 실행하지 않습니다. 복원에 필요한 모델·State·노드·엣지만 다시 정의합니다.

### 그래프 코드를 다시 준비합니다

아래 코드는 1절과 같은 정의입니다. 실행해도 질문을 새로 보내거나 기존 대화를 덮지 않습니다.

In [ ]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

# 앞 단원과 같은 환경 설정을 읽고 답변 모델을 준비
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)


# 대화와 검토 대기를 보관할 SQLite 파일 위치 준비
from pathlib import Path
from langgraph.checkpoint.sqlite import SqliteSaver

material_dir = Path(".")
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
chat_db = output_dir / "lesson03_chat.sqlite"
approval_db = output_dir / "lesson03_approval.sqlite"
print("대화 저장 파일:", chat_db.resolve())
print("승인 저장 파일:", approval_db.resolve())

# 대화마다 유지할 상담 근거와 메시지 누적 규칙 정의
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    policy: str  # 답변의 근거로 유지할 서비스 제작 상담 정책
    messages: Annotated[list[AnyMessage], add_messages]  # 같은 대화의 질문과 답변을 누적

# 정책과 이전 대화를 읽어 새 답변을 만드는 체인과 노드 정의
from IPython.display import Markdown, display
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "서비스 제작 의뢰 상담 담당자입니다. 대화의 마지막 질문에 한국어로 짧게 답하세요. "
     "이전 질문의 서비스 맥락을 유지하고, 지금 확인할 정보를 최대 3개로 정리하세요. "
     "제공 정책에 없는 비용이나 일정을 만들지 말고 해당 정책 ID를 [reservation]처럼 표시하세요. "
     "실제 접수나 계약이 완료되었다고 말하지 마세요.\n정책:\n{policy}"),
    MessagesPlaceholder("messages"),
])
answer_chain = answer_prompt | llm

def answer(state: ChatState):
    """저장된 정책과 대화를 읽고 새 답변 한 개를 반환합니다."""
    reply = answer_chain.invoke({"policy": state["policy"], "messages": state["messages"]})
    return {"messages": [reply]}

from langgraph.graph import END, START, StateGraph

chat_builder = StateGraph(ChatState)
chat_builder.add_node("answer", answer)
# 노드 사이의 실행 흐름 연결
chat_builder.add_edge(START, "answer")
chat_builder.add_edge("answer", END)

# thread_id는 사용자 ID가 아니라 각각의 대화를 구분하는 ID
config_a = {"configurable": {"thread_id": "service-reservation-v1"}}
config_b = {"configurable": {"thread_id": "service-catalog-v1"}}

### 새 입력을 보내기 전에 저장된 내용을 조회합니다

같은 파일과 thread_id를 사용합니다. `get_state()`는 저장 기록을 읽으며 GPT를 호출하지 않습니다.

In [ ]:
# 같은 파일과 대화 ID에서 저장된 상태부터 조회
with SqliteSaver.from_conn_string(str(chat_db)) as checkpointer:
    chat_graph = chat_builder.compile(checkpointer=checkpointer)
    for config in [config_a, config_b]:
        saved = chat_graph.get_state(config)
        print("복원한 대화:", config["configurable"]["thread_id"])
        for message in saved.values.get("messages", []):
            print(message.type + ":")
            display(Markdown(message.text))
        print("다음 노드:", saved.next)
        if not saved.values:
            print("저장 기록이 없습니다. 2절 실행 여부와 파일 경로, thread_id를 확인하세요.")

### A에만 후속 질문을 보냅니다

완료된 대화를 이어갈 때는 **새 메시지가 담긴 입력 딕셔너리**를 보냅니다. 이전 메시지와 policy는 체크포인터에서 읽습니다. `Command(resume=...)`는 사람의 응답을 기다리는 중단을 재개할 때 사용합니다.

아래 코드는 `reservation-followup` ID가 이미 있는지 확인해 같은 질문의 중복 전송을 막습니다. 새 발화에는 새 ID를 사용합니다. thread_id는 대화 전체, 메시지 ID는 그 안의 한 발화를 구분합니다.

교안 02와 같이 `stream(version="v2")`로 변경분을 읽습니다. `type`은 스트림 종류, `data`는 노드가 반환한 내용입니다.

In [ ]:
# 같은 파일과 대화 ID에서 저장된 상태부터 조회
with SqliteSaver.from_conn_string(str(chat_db)) as checkpointer:
    chat_graph = chat_builder.compile(checkpointer=checkpointer)
    saved = chat_graph.get_state(config_a)
    messages = saved.values.get("messages", [])
    if messages and not any(message.id == "reservation-followup" for message in messages):
        question = HumanMessage(content="그 서비스의 견적을 받으려면 추가로 무엇을 정해야 하나요?", id="reservation-followup")
        print("사용자:", question.text)
        # v2의 data에서 답변 노드가 반환한 새 메시지를 읽음
        for update in chat_graph.stream(
            {"messages": [question]}, config_a, stream_mode="updates", version="v2",
        ):
            print("상담 답변:")
            display(Markdown(update["data"]["answer"]["messages"][-1].text))
    for config in [config_a, config_b]:
        current = chat_graph.get_state(config)
        print(config["configurable"]["thread_id"], "메시지 수:", len(current.values.get("messages", [])))

처음 완료했을 때 A는 메시지 4개, B는 2개입니다. A의 후속 답변이 **예약 서비스의 운영 조건**을 묻고, B의 상품 소개 상담과 섞이지 않는지 읽어 보세요. 재실행하면 같은 후속 질문을 다시 호출하지 않습니다.

## 4. 안내문 승인 대기를 저장합니다

상담 담당자가 작성한 안내문을 팀장이 검토하는 상황입니다. 초안과 대기 지점을 저장하면 팀장의 응답 전에 프로그램이 종료되어도 이어갈 수 있습니다. 이 실습은 상담 기록과 승인 기록을 관리하기 쉽게 파일을 나눕니다. State 구조가 다르다고 반드시 파일을 나눠야 하는 것은 아닙니다.

### 4-1. State: 초안과 검토 상태를 담습니다

In [ ]:
# 검토할 초안과 사람의 판단 및 처리 상태를 함께 보관
from typing_extensions import TypedDict

class ApprovalState(TypedDict):
    draft: str  # 사람이 확인할 고객 안내 초안
    decision: str  # 승인 또는 반려로 받은 검토 결과
    status: str  # 검토대기, 안내확정, 수정필요 중 현재 처리 상태

### 4-2. 노드 review: 사람의 판단을 기다립니다

전체 그래프는 `START -> review -> finalize -> END`입니다. review의 `interrupt()`가 초안을 호출자에게 보여 주고 대기합니다. 이 실습에서는 `승인` 또는 `반려`를 응답으로 보냅니다. 선택지 표시는 입력값을 자동 검증하지 않습니다.

In [ ]:
from langgraph.types import Command, interrupt

def review(state: ApprovalState):
    """초안을 보여 주고 사람의 검토 응답을 기다립니다."""
    # 초안을 전달하고 실행을 중단. 재개하면 resume로 보낸 승인·반려 문자열을 반환
    decision = interrupt({"draft": state["draft"], "choices": ["승인", "반려"]})
    return {"decision": decision}

### 노드 finalize: 검토 결과를 상태에 반영합니다

review 다음 작업입니다. 승인은 안내확정, 반려는 수정필요로 기록합니다. 이 예제에서는 외부 메일을 보내지 않고 State만 갱신합니다.

In [ ]:
# 사람의 검토 결과를 안내 상태로 변환
def finalize(state: ApprovalState):
    """검토 결과에 따라 안내 상태를 확정합니다."""
    status = "안내확정" if state["decision"] == "승인" else "수정필요"
    return {"status": status}

### 4-3. 엣지: 검토 후 확정으로 연결합니다

검토 결과에 따라 이동할 노드는 같고 status 값만 달라집니다. 따라서 조건부 엣지 없이 고정 엣지로 연결합니다.

In [ ]:
from langgraph.graph import END, START, StateGraph

approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("review", review)
approval_builder.add_node("finalize", finalize)
# 검토가 끝난 뒤 결과를 확정하는 실행 흐름 연결
approval_builder.add_edge(START, "review")
approval_builder.add_edge("review", "finalize")
approval_builder.add_edge("finalize", END)

### 두 초안을 검토 대기로 저장합니다

첫 초안은 확인 후 견적을 안내하며, 둘째 초안은 확인 없이 제작비를 확정한다고 적혀 있습니다. 다음 절에서 각각 승인과 반려로 재개합니다.

In [ ]:
# 키는 작업 ID, 값은 검토할 안내 초안
approval_drafts = {
    "reservation-approval-v1": "예약 서비스의 운영 시간과 취소 기준을 확인한 뒤 견적을 안내합니다.",
    "catalog-approval-v1": "상품 사진과 설명 자료를 확인하지 않고 제작비를 확정하겠습니다.",
}

In [ ]:
from IPython.display import Image, display

# 초안과 검토 대기를 같은 작업 ID로 저장
with SqliteSaver.from_conn_string(str(approval_db)) as checkpointer:
    approval_graph = approval_builder.compile(checkpointer=checkpointer)
    display(Image(approval_graph.get_graph().draw_mermaid_png()))
    for thread_id, draft in approval_drafts.items():
        config = {"configurable": {"thread_id": thread_id}}
        saved = approval_graph.get_state(config)
        if not saved.values:
            approval_graph.invoke({"draft": draft, "decision": "", "status": "검토대기"}, config)
        saved = approval_graph.get_state(config)
        print("작업:", thread_id, "/ 상태:", saved.values["status"])
        print("다음 노드:", saved.next)
        for item in saved.interrupts:
            print("검토 요청:", item.value)

처음 실행하면 status는 검토대기, next는 `('review',)`입니다. **`next`는 다음에 실행할 노드**, **`interrupts`는 응답을 기다리는 중단 요청**입니다. 사람의 응답을 보낼지는 `saved.interrupts`로 확인합니다. 조회만으로 실행이 재개되지는 않습니다.

## 5. 커널을 재시작하고 승인·반려합니다

**커널을 재시작만 하고 아래 셀부터 실행하세요.** 4절의 최초 입력을 다시 보내지 않습니다. 동일한 State와 노드 이름, 연결을 다시 준비합니다. 이 그래프에는 LLM 노드가 없어 모델 설정도 필요하지 않습니다.

### 저장된 실행을 읽을 코드를 다시 준비합니다

In [ ]:
# 대화와 검토 대기를 보관할 SQLite 파일 위치 준비
from pathlib import Path
from langgraph.checkpoint.sqlite import SqliteSaver

material_dir = Path(".")
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
chat_db = output_dir / "lesson03_chat.sqlite"
approval_db = output_dir / "lesson03_approval.sqlite"
print("대화 저장 파일:", chat_db.resolve())
print("승인 저장 파일:", approval_db.resolve())

# 검토할 초안과 사람의 판단 및 처리 상태를 함께 보관
from typing_extensions import TypedDict

class ApprovalState(TypedDict):
    draft: str  # 사람이 확인할 고객 안내 초안
    decision: str  # 승인 또는 반려로 받은 검토 결과
    status: str  # 검토대기, 안내확정, 수정필요 중 현재 처리 상태

from langgraph.types import Command, interrupt

def review(state: ApprovalState):
    """초안을 보여 주고 사람의 검토 응답을 기다립니다."""
    # 초안을 전달하고 실행을 중단. 재개하면 resume로 보낸 승인·반려 문자열을 반환
    decision = interrupt({"draft": state["draft"], "choices": ["승인", "반려"]})
    return {"decision": decision}

# 사람의 검토 결과를 안내 상태로 변환
def finalize(state: ApprovalState):
    """검토 결과에 따라 안내 상태를 확정합니다."""
    status = "안내확정" if state["decision"] == "승인" else "수정필요"
    return {"status": status}

from langgraph.graph import END, START, StateGraph

approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("review", review)
approval_builder.add_node("finalize", finalize)
# 검토가 끝난 뒤 결과를 확정하는 실행 흐름 연결
approval_builder.add_edge(START, "review")
approval_builder.add_edge("review", "finalize")
approval_builder.add_edge("finalize", END)

### 저장된 초안을 확인하고 같은 thread_id로 재개합니다

`Command(resume=응답)`의 값은 review 노드 안의 `interrupt()` 반환값이 됩니다. `saved.interrupts`가 있을 때만 응답을 보내고, 이미 완료된 작업은 조회만 합니다. 다음 노드가 남았다는 이유만으로 사람의 입력 대기라고 판단하지 않습니다.

In [ ]:
# 키는 기존 작업 ID, 값은 해당 초안을 읽은 사람이 내린 검토 결과
review_decisions = {"reservation-approval-v1": "승인", "catalog-approval-v1": "반려"}
with SqliteSaver.from_conn_string(str(approval_db)) as checkpointer:
    approval_graph = approval_builder.compile(checkpointer=checkpointer)
    for thread_id, decision in review_decisions.items():
        config = {"configurable": {"thread_id": thread_id}}
        saved = approval_graph.get_state(config)
        print("복원한 작업:", thread_id)
        print("복원한 초안:", saved.values.get("draft", "저장 기록 없음"))
        print("재개 전 다음 노드:", saved.next)
        if saved.interrupts:
            approval_graph.invoke(Command(resume=decision), config)
        current = approval_graph.get_state(config)
        print("검토 결과:", current.values.get("decision"))
        print("처리 상태:", current.values.get("status"))
        print("재개 후 다음 노드:", current.next)

예약 안내는 **승인·안내확정**, 상품 안내는 **반려·수정필요**가 되어야 합니다. 둘 다 완료 후 next는 빈 튜플입니다.

재개할 때 **interrupt가 있던 노드는 처음부터 다시 실행**됩니다. 중단 전의 메일 발송 같은 작업은 반복될 수 있습니다. 중단 뒤에 두더라도 장애·재시도에 대비한 업무 ID 기반 중복 방지는 별도로 필요합니다. [공식 중단·재개 문서](https://docs.langchain.com/oss/python/langgraph/interrupts)

### 실무에서 함께 구분할 점

- `SqliteSaver`는 동기식 로컬 실습과 소규모 실행에 적합합니다. 여러 서버가 상태를 공유하는 서비스에서는 PostgreSQL 같은 서버형 체크포인터를 검토합니다.
- thread_id는 인증 정보가 아닙니다. 서비스에서는 로그인한 사용자가 해당 대화에 접근할 수 있는지 확인합니다.
- DB 파일에는 대화·초안과 검토 대기 상태가 저장됩니다. 재시작 후 같은 DB 파일에 연결하고, 같은 `thread_id`로 저장된 실행을 찾습니다.

## 🖐️ 함께 따라하기: 날씨와 검색으로 여행을 계획하고 다시 이어갑니다

**날씨와 여행 정보를 조사해 일정을 만들고, 승인 대기를 SQLite에 저장하는 여행 플래너**입니다. 재시작 후에는 저장한 근거와 일정을 읽고 승인합니다.

<img src="images/09_travel_planner_wiki.png" width="1050" style="background:white" alt="날씨 조회, Wikipedia 검색 Tool을 사용하는 조사 Agent, 일정 작성, 승인 대기와 SQLite 복원 흐름">

| 순서 | 직접 작성할 것 |
|---|---|
| 1 | 여행 조건·근거·계획을 담는 State |
| 2~5 | 날씨·조사·계획·승인 노드의 입력과 반환값 |
| 6 | 노드와 엣지 연결 |
| 7 | SQLite에 계획과 승인 대기 저장 |
| 8 | 커널 재시작 후 조회·승인·결과 저장 |

날씨 HTTP 처리와 검색 Tool·Agent·프롬프트는 제공합니다. 도시·날짜·관심사를 바꾸어도 같은 구조를 사용합니다. 과제 03의 출장 비서는 별도 실습입니다.

### 준비: 모델과 저장 경로를 설정합니다

아래 두 셀은 그대로 실행하세요. 이 따라하기만 시작하거나 커널을 재시작할 때도 필요합니다. 정의를 준비하는 동안에는 외부 API를 호출하지 않습니다.

저장소 루트에서 `uv sync --frozen`을 실행해 필요한 패키지를 설치하세요. 날씨·Wikipedia 검색용 키는 필요 없으며, GPT는 기존 키를 사용합니다.

In [ ]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

# 앞 단원과 같은 환경 설정을 읽고 답변 모델을 준비
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)

라이브러리·DB 경로·작업 ID를 준비하세요. 복원할 때는 같은 파일과 ID를 유지합니다.

In [ ]:
# 여행 그래프에 필요한 도구와 저장 위치를 준비합니다.
import json
from datetime import date, datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import requests
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from langgraph.graph import START, END, StateGraph
from langgraph.types import Command, interrupt
from langgraph.checkpoint.sqlite import SqliteSaver
from IPython.display import Markdown, display

output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
travel_db = output_dir / "lesson03_travel.sqlite"
travel_config = {"configurable": {"thread_id": "travel-plan-v1"}}
print("저장 파일:", travel_db.resolve())

### 1. State: 조건·근거·계획을 담습니다

`TravelState(TypedDict)` 전체를 작성하세요. 아래 필드를 사용합니다.

| 필드 | 자료형 | 담을 내용 |
|---|---|---|
| request | dict | 목적지·좌표·날짜·관심사·예산 |
| weather | dict | 날짜별 예보·조회 시각·실패 이유 |
| research_messages | list[AnyMessage] | 조사 Agent의 검색 호출과 결과 |
| research | str | 조사한 내용의 요약 |
| sources | list[str] | 검색 Tool이 반환한 출처 URL |
| plan / weather_reason | str | 여행 일정 / 날씨 반영 이유 |
| pending_items | list[str] | 아직 확인하지 못한 정보 |
| decision / status | str | 사람의 응답 / 진행 상태 |

이번 그래프는 한 계획의 조사 결과를 한 번 저장합니다. 메시지를 누적하는 대화 그래프와 달리 `add_messages`는 필요하지 않습니다.

In [ ]:
# 여기에 TravelState를 작성하세요

### 2. 날씨 노드: 예보를 조회합니다

아래 함수는 제공 코드입니다. `request`의 좌표·날짜·시간대로 Open-Meteo를 조회하고 **기온(°C)·강수확률(%)·조회 시각**을 반환합니다. 이번 입력에는 확인한 도시 좌표를 제공합니다.

예보 범위 밖이거나 연결이 실패하면 `status="unavailable"`과 이유를 반환합니다. 알 수 없는 날씨를 맑음으로 바꾸지 않습니다. 무료 서버는 비상업 용도 조건이며, 예보 범위는 오늘부터 최대 16일입니다. [API 문서](https://open-meteo.com/en/docs) · [이용 조건](https://open-meteo.com/en/terms)

In [ ]:
# HTTP 처리와 날짜별 결과 정리는 제공 코드입니다.
def fetch_weather(request: dict):
    """여행 날짜의 실제 예보 또는 조회할 수 없는 이유를 반환합니다."""
    checked_at = datetime.now(timezone.utc).isoformat()
    result = {"status": "unavailable", "checked_at": checked_at,
              "source": "https://open-meteo.com/", "daily": [], "reason": ""}
    start = date.fromisoformat(request["start_date"])
    end = date.fromisoformat(request["end_date"])
    today = datetime.now(ZoneInfo(request["timezone"])).date()
    if not today <= start <= end <= today + timedelta(days=15):
        result["reason"] = "여행 기간 전체가 오늘부터 16일 예보 범위 안에 있어야 합니다."
        return result
    params = {
        "latitude": request["latitude"], "longitude": request["longitude"],
        "start_date": start.isoformat(), "end_date": end.isoformat(),
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
        "timezone": request["timezone"],
    }
    try:
        response = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=20)
        response.raise_for_status()
        data = response.json()["daily"]
        for index, day in enumerate(data["time"]):
            result["daily"].append({"date": day, "max_c": data["temperature_2m_max"][index],
                                    "min_c": data["temperature_2m_min"][index],
                                    "rain_percent": data["precipitation_probability_max"][index]})
        expected_days = (end - start).days + 1
        if len(result["daily"]) == expected_days and all(
            value is not None for row in result["daily"] for value in row.values()
        ):
            result["status"] = "ok"
        else:
            result["reason"] = "일부 날짜 또는 예보 값이 누락되었습니다."
    except (requests.RequestException, ValueError, KeyError, IndexError) as error:
        result["reason"] = "날씨 조회 실패: " + type(error).__name__
    return result

`load_weather`에서 State의 `request`를 `fetch_weather`에 전달하세요. 결과를 `weather` 필드로 반환합니다.

In [ ]:
# 날씨 조회 결과를 State에 넣습니다.
def load_weather(state: TravelState):
    weather = ...
    return ...

### 3. 조사 노드: Agent가 검색 Tool을 사용합니다

아래 `@tool` 함수는 한국어 Wikipedia 공식 API에서 **문서 제목·URL·도입부 요약**을 가져옵니다. API 키 없이 장소의 위치·특징을 찾습니다. 현재 전시·운영시간·요금·예약은 별도 확인 사항입니다.

검색어는 도시명과 장소 유형처럼 짧게 씁니다. 실패·빈 결과도 응답에 남기며, 조회 시각은 문서 작성 시각이 아닙니다. [검색 API](https://www.mediawiki.org/wiki/API:Search) · [요약 API](https://www.mediawiki.org/wiki/Extension:TextExtracts#API)

In [ ]:
# 검색과 문서 요약을 한 HTTP 요청으로 가져옵니다. 검색어는 Agent가 만듭니다.
@tool
def search_travel_info(query: str) -> dict:
    """한국어 Wikipedia에서 장소의 소개·위치·특징을 찾습니다. 도시명과 장소 유형 또는 시설명을 짧게 쓰세요. 날짜·site 조건은 쓰지 않습니다."""
    result = {"status": "unavailable", "source": "한국어 Wikipedia", "query": query,
              "checked_at": datetime.now(timezone.utc).isoformat(), "results": [], "reason": ""}
    params = {"action": "query", "format": "json", "formatversion": 2,
              "generator": "search", "gsrsearch": query, "gsrlimit": 3, "gsrnamespace": 0,
              "prop": "extracts|info", "exintro": 1, "explaintext": 1,
              "exchars": 600, "exlimit": 3, "inprop": "url"}
    # 요청을 보내는 프로그램을 식별합니다. 인증 키가 아닙니다.
    headers = {"User-Agent": "EnkoaTravelLesson/1.0 (interactive educational notebook)"}
    try:
        response = requests.get("https://ko.wikipedia.org/w/api.php", params=params,
                                headers=headers, timeout=20)
        response.raise_for_status()
        data = response.json()
        if "error" in data:
            result["reason"] = "Wikipedia API 오류: " + data["error"]["code"]
            return result
        pages = sorted(data.get("query", {}).get("pages", []), key=lambda page: page["index"])
        result["results"] = [{"title": page["title"], "url": page["fullurl"],
                              "summary": page.get("extract", "")} for page in pages]
        result["status"] = "ok" if pages else "empty"
    except requests.HTTPError as error:
        result["reason"] = f"Wikipedia HTTP 오류: {error.response.status_code}"
    except (requests.RequestException, ValueError, KeyError) as error:
        result["reason"] = "Wikipedia 조회 실패: " + type(error).__name__
    return result

제공된 `create_agent`는 모델에 검색 Tool을 연결합니다. **모델의 도구 호출 → 검색 실행 → 결과를 읽은 응답**을 처리합니다. 여행 조건에 맞게 검색어를 정하므로 특정 도시를 프롬프트에 넣지 않습니다.

In [ ]:
# 검색 호출과 결과 읽기를 수행하는 Agent를 준비합니다.
travel_researcher = create_agent(
    model=llm,
    tools=[search_travel_info],
    system_prompt=(
        "여행 자료 조사 담당자입니다. 요청의 목적지와 관심사에 맞춰 search_travel_info를 반드시 사용하세요. "
        "도구는 한국어 Wikipedia 문서를 검색합니다. 관심사를 장소 유형으로 바꾸어 도시명과 장소 유형 또는 시설명으로 짧게 검색하세요. 날짜·site 조건·긴 여행 문장은 넣지 마세요. "
        "서로 다른 관심 분야를 나누어 2회 검색하고, 실패·빈 결과·관련 후보가 없는 분야만 검색어를 바꾸어 1회 더 시도하세요. 전체 검색은 최대 3회입니다. "
        "도구 결과에 나온 장소와 활동을 간결하게 정리하고 출처 URL을 붙이세요. "
        "검색어에 넣은 장소명은 근거가 아닙니다. 제목·요약에서 요청한 목적지와의 관련성이 확인된 장소만 후보로 쓰고 다른 지역의 동명 시설은 제외하세요. 소개가 부족한 후보는 추가 확인 대상으로 남기세요. "
        "Wikipedia의 장소 소개는 방문일의 전시·영업시간·요금·예약을 확인한 자료가 아닙니다. 확인되지 않은 행사명·세부 수치는 모든 출력에서 제외하세요. 검색 실패와 미확보 관심 분야는 밝히고 근거 없는 장소·가격·운영 정보를 만들지 마세요."
    ),
)

`research_destination`에 두 부분을 작성하세요.

- `travel_researcher`에 제공된 `query`를 사용자 메시지로 전달하고 결과를 `response`에 담습니다.
- 반환 필드는 `research_messages`(전체 메시지), `research`(마지막 답변의 text), `sources`(제공 코드가 모은 URL)입니다.

출처는 모델이 임의로 만든 주소를 쓰지 않고, Tool 응답에서 모읍니다.

In [ ]:
# 조사 요청과 출처 추출은 제공하고 Agent 호출·반환을 작성합니다.
def research_destination(state: TravelState):
    query = json.dumps({"request": state["request"], "weather": state["weather"]}, ensure_ascii=False)
    response = ...
    sources = []
    for message in response["messages"]:
        if message.type == "tool":
            packet = json.loads(message.content)
            for row in packet.get("results", []):
                if row["url"] not in sources:
                    sources.append(row["url"])
    return ...

### 4. 계획 노드: 근거를 일정으로 정리합니다

제공 스키마는 **일정·날씨 반영 이유·확인할 사항**으로 출력 형식을 정합니다. 도시·날짜·금액은 필드에 고정하지 않습니다. 구조화 출력이 사실의 정확성까지 보장하지는 않으므로 7단계에서 원자료와 대조합니다.

In [ ]:
# 여행 목적지를 바꾸어도 같은 출력 형식을 사용합니다.
class TravelPlan(BaseModel):
    plan: str = Field(description="요청한 여행 날짜별 활동과 순서, 선택 이유를 담은 간결한 일정")
    weather_reason: str = Field(description="날짜별 실제 예보 수치와 일정·준비물에 반영한 이유. 조회 실패도 명시")
    pending_items: list[str] = Field(description="미확인 운영 정보·비용·예약·조회 실패 등 확인할 사항")

travel_plan_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 여행 플래너입니다. 입력 조건과 조회 근거로 간결한 일정을 작성하세요. "
     "실제 Tool 결과의 제목·요약에서 목적지와의 관련성이 확인된 장소만 사용하세요. 다른 지역의 시설과 검색어·조사 요약에만 나온 장소는 제외하세요. 서로 다른 관심사를 일정에 반영하되 후보가 없는 분야는 미확보라고 표시하고 해당 일정이 미완성임을 밝히세요. Wikipedia는 장소 소개의 근거이며 방문일 운영의 근거는 아닙니다. 날짜별 기온·강수확률과 실내외 선택 이유를 적으세요. "
     "강수확률을 강수 확정으로 말하지 마세요. 예보를 못 읽으면 날씨에 따른 선택은 보류하고 확인할 사항에 쓰세요. "
     "여행 기간과 관심사를 지키고, 예산은 사용자의 한도와 확인된 비용을 구분하세요. "
     "비용이 없으면 합계나 예산 충족을 단정하지 마세요. 운영시간·요금·예약 가능 여부는 미확인으로 남기세요. "
     "개최 기간이 확인되지 않은 행사·전시명과 세부 수치는 모든 출력에서 제외하고 시설만 후보로 쓰세요. "
     "조회 실패 시 임의의 장소로 채우지 말고 미완성 계획임을 밝히세요. 실제 예약이 완료됐다고 쓰지 마세요. "
     "일정은 날짜별 Markdown 목록으로 쓰고 줄바꿈 문자를 사용하세요."),
    ("human", "여행 조건:\n{request}\n날씨 원자료:\n{weather}\n검색 원자료:\n{evidence}\n조사 요약:\n{research}"),
])
travel_plan_chain = travel_plan_prompt | llm.with_structured_output(TravelPlan, strict=True)

`prepare_travel`의 체인 호출과 반환을 작성하세요. 입력은 `request·weather·evidence·research`입니다. JSON 변환·Tool 원자료 수집·일정 줄바꿈 정리는 제공합니다. 응답의 `plan·weather_reason·pending_items`를 동명 State 필드로 반환하고, `status`는 `승인대기`로 설정하세요.

In [ ]:
# 날씨와 검색 원자료를 계획 생성에 전달합니다.
def prepare_travel(state: TravelState):
    evidence = [message.content for message in state["research_messages"] if message.type == "tool"]
    payload = {"request": json.dumps(state["request"], ensure_ascii=False),
               "weather": json.dumps(state["weather"], ensure_ascii=False),
               "evidence": "\n".join(evidence), "research": state["research"]}
    response = ...
    response.plan = response.plan.replace("\\n", "\n")
    return ...

### 5. 승인 노드: 계획을 보여 주고 판단을 기록합니다

`review_travel`은 제공된 요청으로 `interrupt`하고, 응답을 `decision`으로 반환하세요. 날씨와 검색은 앞 노드에서 끝났으므로 승인 재개 시 다시 조회하지 않습니다.

In [ ]:
# 계획과 근거를 보여 주고 승인 또는 반려를 기다립니다.
def review_travel(state: TravelState):
    request = {"plan": state["plan"], "weather_reason": state["weather_reason"],
               "sources": state["sources"], "pending_items": state["pending_items"],
               "choices": ["승인", "반려"]}
    decision = ...
    return ...

`finish_travel` 전체를 작성하세요. 승인이면 `status="계획확정"`, 반려이면 `status="재검토필요"`를 반환합니다. 두 경우 모두 계획 원문은 유지합니다. 계획 승인은 실제 예약 완료를 뜻하지 않습니다.

In [ ]:
# 여기에 finish_travel(state: TravelState)을 작성하세요

### 6. 엣지: 조회·조사·작성·승인을 연결합니다

`TravelState`로 `travel_builder`를 생성하세요. 노드 등록은 제공합니다. 다음 순서로 고정 엣지를 연결하세요.

`START → load_weather → research_destination → prepare_travel → review_travel → finish_travel → END`

검색 Tool 호출은 `research_destination` 안의 Agent가 처리합니다. SQLite 연결은 7단계에서 합니다.

In [ ]:
# StateGraph 생성과 엣지 연결을 완성합니다.
travel_builder = ...
travel_builder.add_node("load_weather", load_weather)
travel_builder.add_node("research_destination", research_destination)
travel_builder.add_node("prepare_travel", prepare_travel)
travel_builder.add_node("review_travel", review_travel)
travel_builder.add_node("finish_travel", finish_travel)
# 여기에 START부터 END까지 엣지를 연결하세요

### 7. 실행: 계획과 승인 대기를 저장합니다

아래 여행 조건은 제공 입력입니다. **첫 실행일의 이틀 뒤부터 1박 2일**로 정합니다. 조건을 바꾸어 새 계획을 만들려면 준비 셀의 작업 ID를 `travel-plan-v2`처럼 바꾸세요. 복원할 때는 같은 ID를 유지합니다.

In [ ]:
# 이 셀은 처음 계획을 만들 때만 실행합니다. 복원할 때는 건너뜁니다.
travel_start = datetime.now(ZoneInfo("Asia/Seoul")).date() + timedelta(days=2)
travel_request = {
    "destination": "부산", "latitude": 35.1796, "longitude": 129.0756,
    "timezone": "Asia/Seoul", "start_date": travel_start.isoformat(),
    "end_date": (travel_start + timedelta(days=1)).isoformat(),
    "preferences": "성인 2명. 바다 산책과 전시 관람을 좋아하고, 비가 오면 실내 활동을 선호한다.",
    "budget": "두 사람의 현지 활동·식비 합계 20만 원 이내 희망. 교통·숙박 제외. 실제 비용은 확인 전.",
}
travel_input = {"request": travel_request, "weather": {}, "research_messages": [],
                "research": "", "sources": [], "plan": "", "weather_reason": "",
                "pending_items": [], "decision": "", "status": "자료수집"}
print(json.dumps(travel_request, ensure_ascii=False, indent=2))

열린 체크포인터로 컴파일하고, `travel_config`의 저장 상태를 조회하세요. 기록이 없을 때만 `travel_input`으로 실행합니다.

첫 실행은 날씨 API와 검색·GPT를 호출합니다. 같은 ID를 다시 실행하면 저장 결과를 읽습니다. 중간에 오류로 멈췄다면 오류 원인을 해결한 뒤 이 셀을 다시 실행하세요. 제공 코드가 마지막 저장 지점에서 남은 작업을 이어갑니다. 승인 대기일 때 `next`는 `('review_travel',)`이고 `interrupts`가 있습니다.

In [ ]:
# 저장 기록을 조회하고 최초 실행만 수행합니다.
with SqliteSaver.from_conn_string(str(travel_db)) as checkpointer:
    travel_graph = ...
    saved = ...
    if not saved.values:
        travel_graph.invoke(travel_input, travel_config)
    elif saved.next and not saved.interrupts:
        # 오류로 멈춘 작업은 초기 입력 없이 마지막 저장 지점부터 이어갑니다.
        travel_graph.invoke(None, travel_config)
    saved = travel_graph.get_state(travel_config)
    print("상태:", saved.values["status"], "/ 다음:", saved.next)
    print("승인 요청 수:", len(saved.interrupts))

다음 제공 셀로 **날씨 원자료 → 검색 호출·응답 → 일정**을 차례로 읽으세요. 날짜·수치·장소가 원자료와 맞는지 확인합니다. 예산 충족·운영 시간·예약을 근거 없이 확정하지 않았는지도 봅니다.

In [ ]:
# 저장된 근거와 생성 결과를 직접 대조합니다.
print("날씨 원자료:")
print(json.dumps(saved.values["weather"], ensure_ascii=False, indent=2))
for message in saved.values["research_messages"]:
    if message.type == "ai" and message.tool_calls:
        print("검색 호출:", message.tool_calls)
    elif message.type == "tool":
        print("검색 응답:", message.content)
display(Markdown(saved.values["plan"]))
display(Markdown(saved.values["weather_reason"]))
print("출처:", saved.values["sources"])
print("확인할 사항:", saved.values["pending_items"])

### 8. 복원: 커널을 재시작하고 승인합니다

노트북을 저장하고 **커널 재시작만** 하세요.

1. 이 따라하기의 **준비 셀 2개와 1~6단계의 모든 코드 셀**을 다시 실행합니다.
2. **7단계는 건너뜁니다.** 여행 날짜를 다시 만들거나 초기 입력을 보내지 않습니다.
3. 아래 셀에서 같은 파일·ID로 조회한 뒤 근거와 계획을 다시 읽습니다.

정의 셀은 외부 API를 호출하지 않습니다. DB에서 읽는 예보는 **저장 당시 예보**이며 최신 예보로 자동 갱신되지 않습니다. 최신 근거로 다시 계획하려면 새 ID로 7단계부터 진행합니다.

In [ ]:
# 재연결한 그래프에서 승인 대기와 계획을 복원합니다.
with SqliteSaver.from_conn_string(str(travel_db)) as checkpointer:
    travel_graph = ...
    restored = ...
    print("상태:", restored.values.get("status", "저장 기록 없음"))
    print("다음:", restored.next)
    print("여행 조건:", json.dumps(restored.values.get("request", {}), ensure_ascii=False, indent=2))
    print("날씨 원자료:", json.dumps(restored.values.get("weather", {}), ensure_ascii=False, indent=2))
    for message in restored.values.get("research_messages", []):
        if message.type == "ai" and message.tool_calls:
            print("검색 호출:", message.tool_calls)
        elif message.type == "tool":
            print("검색 응답:", message.content)
    print("출처:", restored.values.get("sources", []))
    display(Markdown(restored.values.get("plan", "7단계 실행 여부와 경로·ID를 확인하세요.")))
    display(Markdown(restored.values.get("weather_reason", "")))
    print("확인할 사항:", restored.values.get("pending_items", []))

계획을 읽고 `decision`을 승인 또는 반려로 정하세요. 중단 요청이 있을 때만 `Command`에 응답을 담아 재개합니다. `invoke`의 결과를 `final_values`에 담는 부분을 완성하세요.

승인은 `계획확정`, 반려는 `재검토필요`, 완료 후 `next`는 `()`입니다. 이 셀은 날씨·검색·GPT를 다시 호출하지 않습니다. JSON 저장 코드는 제공합니다.

In [ ]:
# 사람의 판단을 반영하고 계획과 출처를 파일로 남깁니다.
decision = "승인"  # 내용 확인 후 필요하면 "반려"로 바꾸세요.
with SqliteSaver.from_conn_string(str(travel_db)) as checkpointer:
    travel_graph = travel_builder.compile(checkpointer=checkpointer)
    saved = travel_graph.get_state(travel_config)
    if decision not in {"승인", "반려"}:
        print("decision은 승인 또는 반려로 입력하세요.")
    elif saved.interrupts:
        final_values = ...
    final = travel_graph.get_state(travel_config)
    print("판단:", final.values.get("decision"), "/ 상태:", final.values.get("status"))
    print("다음:", final.next)
    if final.values and not final.next and final.values.get("status") in {"계획확정", "재검토필요"}:
        export = {key: final.values[key] for key in ["request", "weather", "plan", "weather_reason", "sources", "pending_items", "decision", "status"]}
        result_path = output_dir / (travel_config["configurable"]["thread_id"] + ".json")
        result_path.write_text(json.dumps(export, ensure_ascii=False, indent=2), encoding="utf-8")
        print("저장:", result_path)
    elif final.next and not final.interrupts:
        print("아직 승인 대기에 도달하지 않았습니다. 7단계 실행 오류를 먼저 해결하세요.")

### 결과를 읽습니다

- 저장 전후의 여행 날짜·예보 조회 시각·계획·출처가 같은지 확인하세요.
- 검색 호출과 Tool 응답이 실제로 저장됐는지 확인하세요. 빈 검색 결과는 확인할 사항에 남아야 합니다.
- 승인 뒤 날씨 조회나 계획 생성이 반복되지 않고, 선택한 판단과 상태가 JSON에 기록되는지 확인하세요.

## 정리

| 하려는 일 | 전달할 값 |
|---|---|
| 같은 대화에 새 질문 추가 | 같은 thread_id + 새 입력 딕셔너리 |
| 다른 대화·계획 시작 | 새로운 thread_id + 초기 입력 |
| 재시작 후 저장 내용 조회 | 같은 파일 + 그래프 정의 + thread_id |
| 승인 대기 재개 | 같은 thread_id + Command(resume=응답) |

**저장된 State를 먼저 읽고, 새 입력인지 대기 응답인지 구분해 이어갑니다.** 다음 시간에는 대화가 길어졌을 때의 메모리와 컨텍스트 관리를 배웁니다.